# Description Experiments Notebook

This notebook documents the process of condensing and distilling the `racquet_description` column from a long-winded marketing brief to a concise and focused semantically-rich blurb. Because I'm a bit jaded about paying $17 a month for Claude's subscription, I wanted to avoid having to complete this complex NLP task with API calls. So, in this notebook, I attempt to do this in an at least semi-efficient manner using Claude Chat. Between starting this notebook and actually getting a good output, I rescraped my raw data so I am only including the last run's outputs (which worked quite well!). I will still discuss the previous "experiments" I ran. You can find the prompts I tried in the `prompts` folder. 

## Data loading and imports

In [1]:
# All imports here
from pathlib import Path
import pprint
import pandas as pd

In [2]:
# Define data directory and load data
DATA_DIR = Path().cwd().parent / "data"
cleaned_data = pd.read_csv(DATA_DIR / "interim" / "racquets_cleaned_2026_03_27.csv", index_col=0)
df = cleaned_data.copy()

In [3]:
df.head()

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,racquet_composition,racquet_power_level,...,racquet_length_in,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper
0,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 2026,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,320.0,Graphite,Low-Medium,...,27.0,11.2,12.99,4.0,66.0,24.0,16.0,19.0,46.0,55.0
1,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2026,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,322.0,Graphite,Low-Medium,...,27.0,11.4,12.79,6.0,66.0,22.0,16.0,20.0,46.0,55.0
3,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2-Pack 2026,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,322.0,Graphite,Low-Medium,...,27.0,11.4,12.79,6.0,66.0,22.0,16.0,20.0,46.0,55.0
4,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Plus 2026,2.0,2.0,299.0,One of the games most powerful and spin-friend...,330.0,Graphite,Low-Medium,...,27.5,11.3,12.99,6.0,65.0,24.0,16.0,19.0,46.0,55.0
5,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Team 2026,4.0,1.0,279.0,Updated with improved aerodynamics and better ...,306.0,Graphite,Low-Medium,...,27.0,10.6,13.00,4.0,66.0,24.0,16.0,19.0,44.0,53.0


## Racquet description overview

First, let's examine what the `racquet_description` column looks like. While there is some semantically useful information in there (e.g. "easy learning curve", "easy acceleration", "spin-friendly", etc), there is a lot of fluff too ("in a class by itself", "fabled combination", etc.)

In [4]:
pprint.pprint(df["racquet_description"][0])

('Engineered for speed. With its fabled combination of speed, power, and spin, '
 'the Pure Aero practically stands in a class by itself, not just for its '
 'legendary accomplishments, but also for its easy learning curve and '
 'universal appeal. Like the many versions that came before it, the Pure Aero '
 '2026 features the same basic spec formula that has attracted generations of '
 'players. With its sub-325 swingweight, it offers easy acceleration from the '
 'baseline and quick handling at net, delivering the kind of spin-friendly '
 'targeting that puts the lines in play. For 2026, Babolat re-engineers the '
 'geometry of the shaft to reduce wind drag, giving this speedy racquet an '
 'even more explosive feel, making it even easier to whip up spin, generate '
 'pace, or reset the point with an outstretched flick. Other updates include '
 'the latest version of NF2 Tech, a dampening technology that uses flax fibers '
 'in strategic locations to provide a softer, more arm-friend

I need to explore ways to strip out just the semantically useful information from the `racquet_description` columns. Since this is a relatively small dataset (only ~320 racquets), it's feasible to just pass them through an LLM to remove the marketing fluff.

## Experiments

In this section, I'll try different prompts (and potentially different models) to see what works best for creating a semantically richer `racquet_description` field. I'll start by creating a CSV with just the racquet IDs and descriptions of all racquets that are not null in `racquet_description`. The CSV is saved to the `data/interim` as `racquets_desc_llm_input_03_25_26.csv`.

In [5]:
# Write valid racquet ids and descriptions to a JSON file to pass to LLM
df_valid_desc = df.copy()
df_valid_desc = df_valid_desc[df_valid_desc["racquet_description"].isna() == False]
df_valid_desc[["racquet_id", "racquet_description"]].to_csv(DATA_DIR / "interim" / "racquets_desc_llm_input_2026_03_27.csv")

In [78]:
df_valid_desc[df_valid_desc["racquet_id"] == "RCBAB-BBDWR"]

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,racquet_composition,racquet_power_level,...,racquet_length_in,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper
37,https://www.tennis-warehouse.com/Babolat_Boost...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Boost Drive W,NaN,NaN,119.0,Easy power. Easy Spin. Great Price,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Experiment 1

For the first experiment, I asked ChatGPT to write me an optimized prompt (after describing what I needed it to do) and then used it. I examined its thinking while it was working through the CSV and found that: 

* It deemed 301 rows too many to manually read and process
* Decided to use heavy regex to try and parse out semantically important information
* Created programmatic fallbacks in case it didn't work properly

While this is not really what I wanted, I decided to examine the outputs anyways to see how well this approach fared. To see the prompt I used, reference `prompts/sem_description_v1.txt`. The outputted CSV file is stored at `data/interim/racquets_desc_llm_output_v1_03_25_26.csv`. After examining the output I found that:

* Chat GPT skipped 3 rows because they didn't contain any semantically useful information. All in all, it wasn't too big of a deal as it's trivial to replace these NAs with empty strings along with the other valid NAs. However, this is a significant red mark when considering this approach's validity in a production environment. Before moving forward, I'm going to try a different prompt that more explictly instructs ChatGPT to use its language model on each row and to make sure that each row returns a non-null description (should fill with "" if no useful information).

**Since running this experiment, I have rescraped the data making this output no longer relevant. Since the output itself was not very good, I decided to not try this prompt again.**

### Experiment 2

For the second experiment, I asked ChatGPT to adjust the prompt to ensure that it actually read each description and to only use Python for file I/O and row tracking. I also added a clause specifically instructing it to use empty quotes if it can't find anything to extract. I examined its thinking while it was working through the CSV and found that: 

* It, again, deemed 301 rows too many to manually read and process
* Stopped early

This time ChatGPT flat out refused to completely finish the task as it recognized that it couldn't complete it to the specifications I requested. I decided to ask Gemini if it could do the task and it said yes. So, I used the same prompt and examined its thinking while it was working through the CSV. I found that:

* It seemed to understand the need for internal chunking like I asked
* It appeared to follow instructions through row 50 but then suddenly terminated and said it was having trouble
* After being prompted to think about why it was having trouble and to try again, it explicitly decided to read the CSV in chunks, write the output to a temp file and continue until it got through all 301 racquets
* It explicitly confirmed that it needed to use its language capabilities for the rewrites 
* It made it slightly after (around 200) before crashing again.

To look at the prompt I tried for experiment 2, look at `prompts/sem_description_v2.txt`.

### Experiment 3

For the third experiment, I moved to Claude. First, I discussed the problem with Claude, including my desire to use only the chat interface to rewrite the descriptions in order to avoid paying API costs. Then, I iteratively worked through Claude's reasoning, double checking that it could actually do what it was saying it could do, to develop a full plan on how to actually parse all 301 rows of the input CSV. Then I asked Claude to write an optimized prompt that I would pass in to a new chat session. Once it did that, I gave it the old prompts and explained how they failed. I asked it to review them and think about why they might've failed and if they included any information that would prove helpful in ensuring accuracy and compliance during this time. After Claude made some final edits to the prompt, I entered the prompt into a new chat session. Claude designed the prompt with the failure scenario of it being unable to do the whole CSV in one go in mind but, because of the checks and row reminders, **Claude was able to one-shot the prompt**. I was even able to prompt Claude to combine the batches neatly into one overall CSV file. Overall, I found that:

* Discussing and reasoning with the LLM about its own capabilities and tool use workflows helps to refine its own understanding of how it can approach a problem before actually generating the prompt for itself.
* Claude is able to batch process CSVs within its chat using it's Python tooling as long as you properly design the process by which it batches and how it reasons through it.
* In particular, making Claude remind itself what rows it just parsed and what batch it was on seemed to reduce context degradation. 
* I was worried about the initial instructions being forgotten so I added a small line telling Claude to reference the initial instructions before each batch. I don't know if it did anything but Claude seemed to remember what to do for all 301 rows.

To look at the prompt I used, open `prompts/sem_description_v3.txt`.

In [7]:
# Load Claude output
llm_output = pd.read_csv(DATA_DIR / "interim" / "racquets_desc_llm_output_2026_03_27.csv")

In [8]:
llm_output.head()

,racquet_id,cleaned_description
0,RCBAB-BPAR26,Sub-325 swingweight delivers easy acceleration...
1,RCBAB-BPA98R,"The smallest head, tightest string spacing, an..."
2,RCBAB-BPA982,Same specs as the Pure Aero 98R (two-racquet p...
3,RCBAB-BPAPLR,"Extended 27.5"" length boosts power and spin po..."
4,RCBAB-BPAT26,"A lighter, faster version of the standard Pure..."


In [ ]:
# Cleaned description
pprint.pprint(llm_output["cleaned_description"].iloc[0])

('Sub-325 swingweight delivers easy acceleration and quick net play with '
 'spin-friendly targeting. The aerodynamic shaft reduces wind drag for an '
 'explosive, whippy feel through contact. FSI Spin Technology with Woofer '
 'grommets increases dwell time and string movement for heavy spin. '
 'Arm-friendly flax fiber dampening (NF2 Tech) softens impact. Suits '
 'aggressive players who want speed, spin, and power.')


In [ ]:
# Original description - much longer, way more "fluff" terms ("fabled combination", "legendary accomplishments", "that puts the lines in play")
pprint.pprint(df[df["racquet_id"] == llm_output["racquet_id"].iloc[0]]["racquet_description"][0])

('Engineered for speed. With its fabled combination of speed, power, and spin, '
 'the Pure Aero practically stands in a class by itself, not just for its '
 'legendary accomplishments, but also for its easy learning curve and '
 'universal appeal. Like the many versions that came before it, the Pure Aero '
 '2026 features the same basic spec formula that has attracted generations of '
 'players. With its sub-325 swingweight, it offers easy acceleration from the '
 'baseline and quick handling at net, delivering the kind of spin-friendly '
 'targeting that puts the lines in play. For 2026, Babolat re-engineers the '
 'geometry of the shaft to reduce wind drag, giving this speedy racquet an '
 'even more explosive feel, making it even easier to whip up spin, generate '
 'pace, or reset the point with an outstretched flick. Other updates include '
 'the latest version of NF2 Tech, a dampening technology that uses flax fibers '
 'in strategic locations to provide a softer, more arm-friend

From the two above cells, we can see that Claude did a pretty decent job at condensing and distill these marketing briefs. Since it would time prohibitive to review all of 322 of these, I am instead going to manually a review a random sample of 10 to make sure that I generally approve of the distillations. But, before doing that, I need to run some basic checks.

In [23]:
assert len(df_valid_desc) == len(llm_output)
assert df_valid_desc["racquet_id"].nunique() == llm_output["racquet_id"].nunique()

AssertionError: 

#### Investigating racquet_id uniqueness mismatch

Here I dive into why there is a mismatch. **Spoiler alert**: Claude mistakenly replaced on racquet's id with the id of another racquet (fair mistake, the racquet was the exact same except 10g heavier). Since it was only one mistake and it was low cost to manually fix it, I just ended up manually changing the ID. If this part of the process were going to be automated, I would recommend asking claude to explictly check and make sure that the output CSV has the same number of unique racquet_id values.

In [48]:
llm_output["racquet_id"].nunique()

322

In [30]:
print(df[df.duplicated('racquet_id', keep=False)])

Empty DataFrame
Columns: [racquet_url, racquet_img, racquet_name, racquet_rating, racquet_rating_count, racquet_price, racquet_description, racquet_swingweight, racquet_composition, racquet_power_level, racquet_stroke_style, racquet_swing_speed, racquet_colors, racquet_grip_type, racquet_brand, racquet_id, racquet_head_size_sq_in, racquet_length_in, racquet_strung_weight_oz, racquet_balance_in, racquet_balance_HH_HL, racquet_stiffness, racquet_avg_beam_width, racquet_mains, racquet_crosses, racquet_tension_lower, racquet_tension_upper]
Index: []

[0 rows x 27 columns]


In [31]:
print(df_valid_desc[df_valid_desc.duplicated('racquet_id', keep=False)])

Empty DataFrame
Columns: [racquet_url, racquet_img, racquet_name, racquet_rating, racquet_rating_count, racquet_price, racquet_description, racquet_swingweight, racquet_composition, racquet_power_level, racquet_stroke_style, racquet_swing_speed, racquet_colors, racquet_grip_type, racquet_brand, racquet_id, racquet_head_size_sq_in, racquet_length_in, racquet_strung_weight_oz, racquet_balance_in, racquet_balance_HH_HL, racquet_stiffness, racquet_avg_beam_width, racquet_mains, racquet_crosses, racquet_tension_lower, racquet_tension_upper]
Index: []

[0 rows x 27 columns]


In [32]:
print(llm_output[llm_output.duplicated('racquet_id', keep=False)])

         racquet_id                                cleaned_description
265  RCVOLKL-VOSV10  Vostra V10 320g: control-oriented with a high ...
266  RCVOLKL-VOSV10  Vostra V10 300g: defined by speed and spin-fri...


In [36]:
df_valid_desc[df_valid_desc["racquet_id"] == "RCVOLKL-VOSV10"]

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,racquet_composition,racquet_power_level,...,racquet_length_in,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper
377,https://www.tennis-warehouse.com/Volkl_Vostra_...,https://img.tennis-warehouse.com/watermark/rs....,Volkl Vostra V10 300g,NaN,NaN,249.99,Speed. Spin. Precision Introducing the Vostra ...,314.0,Red Cell/Graphite,Low-Medium,...,27.0,11.2,13.18,3.0,68.0,21.333333,16.0,19.0,50.0,60.0


In [38]:
df_valid_desc[df_valid_desc["racquet_name"].str.contains("Volkl Vostra V10")][["racquet_name", "racquet_id"]]

,racquet_name,racquet_id
376,Volkl Vostra V10 320g,RCVOLKL-VOSV1B
377,Volkl Vostra V10 300g,RCVOLKL-VOSV10


It appears that Claude made **one** mistake. It accidentally made the racquet_id for the Volkl Vostra V10 320g the same as the racquet_id for the Volkl Vostra V10 300g. Since this is the only duplicate and the only mistake, I think it's fine to just manually fix the mistake. In the future, I should add a uniqueness check on the racquet_ids into the Claude prompt. 

In [ ]:
llm_output.iloc[265] # idx 265 is where the error is

racquet_id                                                RCVOLKL-VOSV10
cleaned_description    Vostra V10 320g: control-oriented with a high ...
Name: 265, dtype: object

In [43]:
# Manually change
llm_output.loc[265, "racquet_id"] = "RCVOLKL-VOSV1B"

In [46]:
# Check if that fixed the uniqueness issue
df_valid_desc["racquet_id"].nunique(), llm_output["racquet_id"].nunique()

(322, 322)

In [ ]:
# How many did Claude skip -> 27 -> not bad
llm_output.isna().sum()

racquet_id              0
cleaned_description    27
dtype: int64

In [ ]:
# Look at a random one that Claude skipped -> fair enough skip, could have said comfortable but I think it's fine.
pprint.pprint(
    df_valid_desc[df_valid_desc["racquet_id"] == (llm_output[llm_output["cleaned_description"].isna()]
                                              .sample(1, random_state=100)["racquet_id"].item())]["racquet_description"].item()
)

'Wilson brings extra comfort to the explosive mechanics of the modern game'


In [71]:
# Keep rerunning to see different random descriptions of "skipped" racquets
pprint.pprint(
    df_valid_desc[df_valid_desc["racquet_id"] == (llm_output[llm_output["cleaned_description"].isna()]
                                              .sample(1)["racquet_id"].item())]["racquet_description"].item()
)

('Updated with more speed for 2024, this ultra-comfortable racquet delivers '
 'easy acceleration from the baseline and quick handling at net')


## Join cleaned text to original data

Now, I join the Claude-cleaned text to the original data. Recall that some of the racquets don't have a description and others yet will have no match on the llm_output data because claude skipped them. I'll use a left join to keep everything from the original data and then I'll handle the missing values in the joint df.

In [72]:
df_joined = df.merge(right = llm_output, how = "left", on = "racquet_id")

In [76]:
df.shape, llm_output.shape, df_joined.shape

((335, 27), (322, 2), (335, 28))

In [77]:
df_joined[df_joined["cleaned_description"].isna()]

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,racquet_composition,racquet_power_level,...,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper,cleaned_description
19,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Drive Super Lite 2025,NaN,NaN,269.00,Introducing the perfect entry point into the w...,295.0,Graphite,Low-Medium,...,9.5,13.19,2.0,68.0,24.000000,16.0,19.0,44.0,53.0,NaN
33,https://www.tennis-warehouse.com/Babolat_Boost...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Boost Drive,NaN,NaN,119.00,Easy power. Easy Spin. Great Price,312.0,Graphite,Medium,...,9.7,13.85,-3.0,65.0,24.000000,16.0,19.0,50.0,55.0,NaN
34,https://www.tennis-warehouse.com/Babolat_Boost...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Boost Drive W,NaN,NaN,119.00,Easy power. Easy Spin. Great Price,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,https://www.tennis-warehouse.com/Babolat_Evo_D...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Evo Drive,5.0,1.0,199.00,Easy power. Easy spin. Great price.,292.0,Graphite,Low-Medium,...,10.7,13.00,4.0,64.0,24.000000,16.0,19.0,50.0,55.0,NaN
36,https://www.tennis-warehouse.com/Babolat_Evo_D...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Evo Drive Lite,5.0,1.0,199.00,User-friendly speed. Easy power. Outstanding p...,290.0,Graphite,Low-Medium,...,9.5,13.00,4.0,62.0,24.000000,16.0,17.0,50.0,55.0,NaN
37,https://www.tennis-warehouse.com/Babolat_Evo_D...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Evo Drive Lite White,NaN,NaN,199.00,User-friendly speed. Easy power. Outstanding p...,290.0,Graphite,Low-Medium,...,9.5,13.00,4.0,62.0,24.000000,16.0,17.0,50.0,55.0,NaN
38,https://www.tennis-warehouse.com/Babolat_EVO_S...,https://img.tennis-warehouse.com/watermark/rs....,Babolat EVO Strike,4.0,1.0,199.00,"Babolat delivers impressive speed, precision a...",309.0,Graphite,Low-Medium,...,10.7,13.00,4.0,66.0,23.500000,16.0,19.0,50.0,55.0,NaN
39,https://www.tennis-warehouse.com/Babolat_Boost...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Boost Wimbledon,4.5,2.0,129.00,Easy Power. Spin-friendly Targeting. Outstandi...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,https://www.tennis-warehouse.com/Wilson_Clash_...,https://img.tennis-warehouse.com/watermark/rs....,Wilson Clash 108 v3,4.3,6.0,299.00,Wilson adds another chapter to the most forgiv...,312.0,Graphite,Medium,...,10.4,13.50,1.0,58.0,24.500000,16.0,19.0,50.0,60.0,NaN
63,https://www.tennis-warehouse.com/Wilson_Clash_...,https://img.tennis-warehouse.com/watermark/rs....,Wilson Clash 100 Pro v2,4.3,22.0,109.00,Wilson brings extra comfort to the explosive m...,325.0,Graphite,Low-Medium,...,11.5,12.40,9.0,59.0,24.500000,16.0,20.0,50.0,60.0,NaN


In [80]:
df_joined[(df_joined["racquet_description"].isna()) & (df_joined["cleaned_description"].isna())]

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,racquet_composition,racquet_power_level,...,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper,cleaned_description
87,https://www.tennis-warehouse.com/Wilson_Triad_...,https://img.tennis-warehouse.com/watermark/rs....,Wilson Triad Five,4.9,10.0,269.00,NaN,324.0,Graphite,Medium,...,10.0,14.37,-6.0,NaN,26.000000,16.0,20.0,50.0,60.0,NaN
136,https://www.tennis-warehouse.com/Head_Radical_...,https://img.tennis-warehouse.com/watermark/rs....,Head Radical Pro 2021,4.2,18.0,139.00,NaN,330.0,Graphene 360+/Graphite,Low-Medium,...,11.7,12.75,6.0,65.0,20.833333,16.0,19.0,48.0,57.0,NaN
207,https://www.tennis-warehouse.com/Prince_Warrio...,https://img.tennis-warehouse.com/watermark/rs....,Prince Warrior 100 (300g),4.7,13.0,99.00,NaN,311.0,Graphite,Low-Medium,...,11.2,12.79,6.0,65.0,24.000000,16.0,19.0,50.0,60.0,NaN
209,https://www.tennis-warehouse.com/Prince_Warrio...,https://img.tennis-warehouse.com/watermark/rs....,Prince Warrior 107 (275g),4.6,20.0,99.00,NaN,313.0,Graphite,Low-Medium,...,10.2,13.50,0.0,68.0,26.333333,16.0,19.0,50.0,60.0,NaN
215,https://www.tennis-warehouse.com/Prince_Twistp...,https://img.tennis-warehouse.com/watermark/rs....,Prince Twistpower X100,4.6,11.0,179.00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
216,https://www.tennis-warehouse.com/Prince_Twistp...,https://img.tennis-warehouse.com/watermark/rs....,Prince Twistpower X105 (290g),4.7,13.0,179.00,NaN,312.0,Graphite,Low-Medium,...,10.7,12.59,7.0,63.0,24.666667,16.0,19.0,45.0,55.0,NaN
237,https://www.tennis-warehouse.com/Tecnifibre_TF...,https://img.tennis-warehouse.com/watermark/rs....,Tecnifibre TF-X1 285,4.7,6.0,119.00,NaN,316.0,Graphite,Low-Medium,...,10.5,13.25,2.0,71.0,24.000000,16.0,19.0,49.0,55.0,NaN
238,https://www.tennis-warehouse.com/Tecnifibre_TF...,https://img.tennis-warehouse.com/watermark/rs....,Tecnifibre TF-X1 275,4.0,5.0,119.00,NaN,310.0,Graphite,Low-Medium,...,10.2,13.38,1.0,69.0,25.000000,16.0,19.0,49.0,55.0,NaN
296,https://www.tennis-warehouse.com/Volkl_V-Cell_...,https://img.tennis-warehouse.com/watermark/rs....,Volkl V-Cell V1 Pro,4.7,7.0,129.00,NaN,316.0,VCell/Graphite,Low-Medium,...,NaN,13.00,4.0,68.0,22.000000,16.0,19.0,50.0,60.0,NaN
297,https://www.tennis-warehouse.com/Volkl_V-Cell_...,https://img.tennis-warehouse.com/watermark/rs....,Volkl V-Cell V1 MP,5.0,5.0,129.00,NaN,302.0,VCell/Graphite,Low-Medium,...,10.6,12.99,4.0,68.0,25.000000,16.0,19.0,50.0,60.0,NaN
